# Notebook 9: Final Summary & Comprehensive Comparison
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

This notebook aggregates results from all experiments and generates comprehensive comparison visualizations for the research paper.

**Experiments Covered:**
1. Same-stock prediction (80/20 and 70/30)
2. Cross-stock prediction (80/20 and 70/30)
3. Different timeframe training (80/20 and 70/30)
4. Multi-stock training (80/20 and 70/30)

**Key Comparisons:**
- BiLSTM vs BiGRU (main comparison)
- BiLSTM/BiGRU vs LSTM/GRU (bidirectional vs unidirectional)
- 80/20 vs 70/30 split performance
- Cross-stock generalization ability
- Timeframe transfer learning


In [2]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

set_ieee_style()

os.makedirs('figures/summary', exist_ok=True)
os.makedirs('results', exist_ok=True)

print("Setup complete!")


stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 60, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
Setup complete!


## 1. Load All Experiment Results

In [3]:
# ============================================================
# LOAD ALL RESULTS CSVs
# ============================================================
result_files = {
    'Exp1_80_20': 'results/Exp1_80_20_results.csv',
    'Exp1_70_30': 'results/Exp1_70_30_results.csv',
    'Exp2_80_20': 'results/Exp2_80_20_results.csv',
    'Exp2_70_30': 'results/Exp2_70_30_results.csv',
    'Exp3_80_20': 'results/Exp3_80_20_results.csv',
    'Exp3_70_30': 'results/Exp3_70_30_results.csv',
    'Exp4_80_20': 'results/Exp4_80_20_results.csv',
    'Exp4_70_30': 'results/Exp4_70_30_results.csv',
}

results = {}
for key, filepath in result_files.items():
    if os.path.exists(filepath):
        results[key] = pd.read_csv(filepath)
        print(f"  Loaded: {key} ({len(results[key])} rows)")
    else:
        print(f"  WARNING: {filepath} not found. Run the corresponding notebook first.")

print(f"\nLoaded {len(results)} result files.")


  Loaded: Exp1_80_20 (16 rows)
  Loaded: Exp1_70_30 (16 rows)
  Loaded: Exp2_80_20 (48 rows)
  Loaded: Exp2_70_30 (48 rows)
  Loaded: Exp3_80_20 (48 rows)
  Loaded: Exp3_70_30 (48 rows)
  Loaded: Exp4_80_20 (16 rows)
  Loaded: Exp4_70_30 (16 rows)

Loaded 8 result files.


## 2. Experiment 1: Same-Stock Prediction — 80/20 vs 70/30

In [4]:
# ============================================================
# COMPARE 80/20 vs 70/30 for Same-Stock Prediction
# ============================================================
if 'Exp1_80_20' in results and 'Exp1_70_30' in results:
    df_80 = results['Exp1_80_20'].copy()
    df_80['Split'] = '80/20'
    df_70 = results['Exp1_70_30'].copy()
    df_70['Split'] = '70/30'
    
    combined = pd.concat([df_80, df_70], ignore_index=True)
    
    # Print comparison
    print("=" * 80)
    print("  EXPERIMENT 1: Same-Stock Prediction - 80/20 vs 70/30")
    print("=" * 80)
    
    for stock in STOCKS:
        print(f"\n--- {stock} ---")
        stock_data = combined[combined['Stock'] == stock]
        pivot = stock_data.pivot_table(
            values=['RMSE', 'MAE', 'MAPE (%)', 'R2'],
            index='Model', columns='Split'
        )
        print(pivot.round(4).to_string())
    
    # Grouped bar chart: RMSE comparison
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Experiment 1: Same-Stock Prediction\nRMSE — 80/20 vs 70/30', 
                 fontsize=16, fontweight='bold')
    
    for idx, stock in enumerate(STOCKS):
        ax = axes[idx // 2, idx % 2]
        stock_data = combined[combined['Stock'] == stock]
        
        x = np.arange(len(MODEL_TYPES))
        w = 0.35
        
        vals_80 = [stock_data[(stock_data['Model'] == m) & (stock_data['Split'] == '80/20')]['RMSE'].values
                   for m in MODEL_TYPES]
        vals_70 = [stock_data[(stock_data['Model'] == m) & (stock_data['Split'] == '70/30')]['RMSE'].values
                   for m in MODEL_TYPES]
        
        vals_80 = [v[0] if len(v) > 0 else 0 for v in vals_80]
        vals_70 = [v[0] if len(v) > 0 else 0 for v in vals_70]
        
        bars1 = ax.bar(x - w/2, vals_80, w, label='80/20', color='#0072B2', alpha=0.8)
        bars2 = ax.bar(x + w/2, vals_70, w, label='70/30', color='#D55E00', alpha=0.8)
        
        ax.set_title(f'{stock}', fontsize=14)
        ax.set_xticks(x)
        ax.set_xticklabels(MODEL_TYPES, fontsize=11)
        ax.set_ylabel('RMSE', fontsize=12)
        ax.legend(fontsize=10)
        
        # Value labels
        for bar in bars1:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                    f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                    f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
    
    fig.tight_layout()
    save_fig(fig, 'figures/summary/exp1_rmse_80vs70.png')
    
    # R² comparison
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Experiment 1: Same-Stock Prediction\nR² Score — 80/20 vs 70/30',
                 fontsize=16, fontweight='bold')
    
    for idx, stock in enumerate(STOCKS):
        ax = axes[idx // 2, idx % 2]
        stock_data = combined[combined['Stock'] == stock]
        
        x = np.arange(len(MODEL_TYPES))
        w = 0.35
        
        vals_80 = [stock_data[(stock_data['Model'] == m) & (stock_data['Split'] == '80/20')]['R2'].values
                   for m in MODEL_TYPES]
        vals_70 = [stock_data[(stock_data['Model'] == m) & (stock_data['Split'] == '70/30')]['R2'].values
                   for m in MODEL_TYPES]
        
        vals_80 = [v[0] if len(v) > 0 else 0 for v in vals_80]
        vals_70 = [v[0] if len(v) > 0 else 0 for v in vals_70]
        
        bars1 = ax.bar(x - w/2, vals_80, w, label='80/20', color='#0072B2', alpha=0.8)
        bars2 = ax.bar(x + w/2, vals_70, w, label='70/30', color='#D55E00', alpha=0.8)
        
        ax.set_title(f'{stock}', fontsize=14)
        ax.set_xticks(x)
        ax.set_xticklabels(MODEL_TYPES, fontsize=11)
        ax.set_ylabel('R² Score', fontsize=12)
        ax.legend(fontsize=10)
    
    fig.tight_layout()
    save_fig(fig, 'figures/summary/exp1_r2_80vs70.png')
    
    print("\nExp1 comparison plots saved!")
else:
    print("Missing Exp1 results. Run Notebooks 1 and 5 first.")


  EXPERIMENT 1: Same-Stock Prediction - 80/20 vs 70/30

--- TLKM ---
             MAE          MAPE (%)              R2              RMSE          
Split      70/30    80/20    70/30   80/20   70/30   80/20     70/30     80/20
Model                                                                         
BiGRU    57.5024  57.8768   1.9577  1.9099  0.9732  0.9661   75.1952   76.0569
BiLSTM  100.0160  61.7840   3.2762  2.0469  0.9321  0.9619  119.6239   80.6680
GRU      63.5888  75.4171   2.1416  2.4338  0.9691  0.9519   80.6747   90.6087
LSTM     74.9748  89.8112   2.5497  2.9133  0.9572  0.9321   95.0013  107.7114

--- BBCA ---
             MAE           MAPE (%)              R2              RMSE          
Split      70/30     80/20    70/30   80/20   70/30   80/20     70/30     80/20
Model                                                                          
BiGRU   324.9464  163.9348   4.2954  2.0120  0.9393  0.9613  389.2240  204.6505
BiLSTM  233.9387  197.9622   3.0760  2.3967 

## 3. BiLSTM vs BiGRU — Main Comparison

In [5]:
# ============================================================
# MAIN COMPARISON: BiLSTM vs BiGRU across ALL experiments
# ============================================================
if 'Exp1_80_20' in results:
    # Filter to only BiLSTM and BiGRU
    bilstm_bigru = results['Exp1_80_20'][
        results['Exp1_80_20']['Model'].isin(['BiLSTM', 'BiGRU'])
    ].copy()
    
    print("=" * 60)
    print("  BiLSTM vs BiGRU — Same-Stock Prediction (80/20)")
    print("=" * 60)
    
    pivot = bilstm_bigru.pivot_table(
        values=['MSE', 'RMSE', 'MAE', 'MAPE (%)', 'R2'],
        index='Stock', columns='Model'
    )
    print(pivot.round(4).to_string())
    
    # Radar/Spider chart for BiLSTM vs BiGRU
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    fig.suptitle('BiLSTM vs BiGRU — Per Stock Performance (80/20)',
                 fontsize=16, fontweight='bold')
    
    metrics_list = ['RMSE', 'MAE', 'MAPE (%)', 'R2']
    
    for idx, stock in enumerate(STOCKS):
        ax = axes[idx // 2, idx % 2]
        
        bilstm_data = bilstm_bigru[
            (bilstm_bigru['Stock'] == stock) & (bilstm_bigru['Model'] == 'BiLSTM')
        ]
        bigru_data = bilstm_bigru[
            (bilstm_bigru['Stock'] == stock) & (bilstm_bigru['Model'] == 'BiGRU')
        ]
        
        x = np.arange(len(metrics_list))
        w = 0.35
        
        bilstm_vals = [bilstm_data[m].values[0] if len(bilstm_data) > 0 else 0 for m in metrics_list]
        bigru_vals = [bigru_data[m].values[0] if len(bigru_data) > 0 else 0 for m in metrics_list]
        
        ax.bar(x - w/2, bilstm_vals, w, label='BiLSTM', color=MODEL_COLORS['BiLSTM'])
        ax.bar(x + w/2, bigru_vals, w, label='BiGRU', color=MODEL_COLORS['BiGRU'])
        
        ax.set_title(f'{stock}', fontsize=14)
        ax.set_xticks(x)
        ax.set_xticklabels(metrics_list, fontsize=10)
        ax.legend(fontsize=10)
    
    fig.tight_layout()
    save_fig(fig, 'figures/summary/bilstm_vs_bigru_same_stock.png')
    print("BiLSTM vs BiGRU comparison saved!")


  BiLSTM vs BiGRU — Same-Stock Prediction (80/20)
            MAE           MAPE (%)                 MSE                  R2              RMSE          
Model     BiGRU    BiLSTM    BiGRU  BiLSTM       BiGRU      BiLSTM   BiGRU  BiLSTM     BiGRU    BiLSTM
Stock                                                                                                 
ASII    73.7760   88.9684   1.5390  1.8398   9678.1743  13595.2684  0.9738  0.9632   98.3777  116.5987
BBCA   163.9348  197.9622   2.0120  2.3967  41881.8134  56805.1783  0.9613  0.9474  204.6505  238.3384
TLKM    57.8768   61.7840   1.9099  2.0469   5784.6547   6507.3259  0.9661  0.9619   76.0569   80.6680
UNVR    98.0873  125.4401   3.8596  5.0301  15751.6179  23875.2842  0.9819  0.9726  125.5054  154.5163
  Figure saved: figures/summary/bilstm_vs_bigru_same_stock.png
BiLSTM vs BiGRU comparison saved!


## 4. Bidirectional vs Unidirectional Comparison

In [6]:
# ============================================================
# BIDIRECTIONAL vs UNIDIRECTIONAL
# ============================================================
if 'Exp1_80_20' in results:
    df = results['Exp1_80_20'].copy()
    df['Direction'] = df['Model'].apply(
        lambda x: 'Bidirectional' if x.startswith('Bi') else 'Unidirectional'
    )
    
    print("=" * 60)
    print("  Bidirectional vs Unidirectional — Average Across Stocks")
    print("=" * 60)
    
    dir_avg = df.groupby(['Direction', 'Model'])[['RMSE', 'MAE', 'MAPE (%)', 'R2']].mean()
    print(dir_avg.round(4).to_string())
    
    # Bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Bidirectional vs Unidirectional RNN — Average Performance',
                 fontsize=16, fontweight='bold')
    
    avg_by_model = df.groupby('Model')[['RMSE', 'R2']].mean()
    
    # RMSE
    ax = axes[0]
    models = MODEL_TYPES
    rmse_vals = [avg_by_model.loc[m, 'RMSE'] for m in models]
    bars = ax.bar(models, rmse_vals, color=[MODEL_COLORS[m] for m in models], 
                  edgecolor='white', linewidth=0.5)
    ax.set_ylabel('RMSE (avg)', fontsize=13)
    ax.set_title('Average RMSE', fontsize=14)
    for bar, val in zip(bars, rmse_vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{val:.2f}', ha='center', va='bottom', fontsize=10)
    
    # R²
    ax = axes[1]
    r2_vals = [avg_by_model.loc[m, 'R2'] for m in models]
    bars = ax.bar(models, r2_vals, color=[MODEL_COLORS[m] for m in models],
                  edgecolor='white', linewidth=0.5)
    ax.set_ylabel('R² Score (avg)', fontsize=13)
    ax.set_title('Average R² Score', fontsize=14)
    for bar, val in zip(bars, r2_vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)
    
    fig.tight_layout()
    save_fig(fig, 'figures/summary/bidirectional_vs_unidirectional.png')
    print("Direction comparison saved!")


  Bidirectional vs Unidirectional — Average Across Stocks
                           RMSE       MAE  MAPE (%)      R2
Direction      Model                                       
Bidirectional  BiGRU   126.1476   98.4187    2.3301  0.9708
               BiLSTM  147.5304  118.5387    2.8284  0.9613
Unidirectional GRU     135.7246  106.7105    2.1844  0.9630
               LSTM    126.8038   96.1343    2.2404  0.9647
  Figure saved: figures/summary/bidirectional_vs_unidirectional.png
Direction comparison saved!


## 5. Cross-Stock Generalization Analysis

In [7]:
# ============================================================
# CROSS-STOCK GENERALIZATION ANALYSIS
# ============================================================
if 'Exp2_80_20' in results:
    df = results['Exp2_80_20'].copy()
    
    print("=" * 60)
    print("  Cross-Stock Generalization — Average RMSE per Model")
    print("=" * 60)
    avg_cross = df.groupby('Model')[['RMSE', 'MAE', 'MAPE (%)', 'R2']].mean()
    print(avg_cross.round(4).to_string())
    
    # Compare same-stock vs cross-stock performance
    if 'Exp1_80_20' in results:
        same_avg = results['Exp1_80_20'].groupby('Model')['RMSE'].mean()
        cross_avg = df.groupby('Model')['RMSE'].mean()
        
        fig, ax = plt.subplots(figsize=(10, 6))
        x = np.arange(len(MODEL_TYPES))
        w = 0.35
        
        same_vals = [same_avg[m] for m in MODEL_TYPES]
        cross_vals = [cross_avg[m] for m in MODEL_TYPES]
        
        ax.bar(x - w/2, same_vals, w, label='Same-Stock (Exp1)', color='#0072B2', alpha=0.8)
        ax.bar(x + w/2, cross_vals, w, label='Cross-Stock (Exp2)', color='#D55E00', alpha=0.8)
        
        ax.set_xticks(x)
        ax.set_xticklabels(MODEL_TYPES, fontsize=12)
        ax.set_ylabel('Average RMSE', fontsize=13)
        ax.set_title('Same-Stock vs Cross-Stock Prediction\nAverage RMSE (80/20)',
                     fontsize=14)
        ax.legend(fontsize=12)
        fig.tight_layout()
        save_fig(fig, 'figures/summary/same_vs_cross_stock_rmse.png')
        print("Same vs Cross-stock comparison saved!")


  Cross-Stock Generalization — Average RMSE per Model
            RMSE       MAE  MAPE (%)      R2
Model                                       
BiGRU   124.4458   96.9698    2.3227  0.9649
BiLSTM  144.6355  110.7610    2.5439  0.9550
GRU     108.2351   81.2864    1.8675  0.9755
LSTM    131.4672  100.8120    2.1760  0.9655
  Figure saved: figures/summary/same_vs_cross_stock_rmse.png
Same vs Cross-stock comparison saved!


## 6. Multi-Stock Training Analysis

In [8]:
# ============================================================
# MULTI-STOCK TRAINING ANALYSIS
# ============================================================
if 'Exp4_80_20' in results:
    df = results['Exp4_80_20'].copy()
    
    print("=" * 60)
    print("  Multi-Stock Training — Results")
    print("=" * 60)
    print(df.to_string(index=False))
    
    # Compare Exp1 (same-stock) vs Exp4 (multi-stock)
    if 'Exp1_80_20' in results:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle('Same-Stock (Exp1) vs Multi-Stock Training (Exp4)\nRMSE Comparison (80/20)',
                     fontsize=16, fontweight='bold')
        
        for idx, target in enumerate(STOCKS):
            ax = axes[idx // 2, idx % 2]
            
            # Same-stock RMSE
            same = results['Exp1_80_20'][results['Exp1_80_20']['Stock'] == target]
            # Multi-stock RMSE
            multi = df[df['Target_Stock'] == target]
            
            x = np.arange(len(MODEL_TYPES))
            w = 0.35
            
            s_vals = [same[same['Model'] == m]['RMSE'].values[0] if len(same[same['Model'] == m]) > 0 else 0 
                      for m in MODEL_TYPES]
            m_vals = [multi[multi['Model'] == m]['RMSE'].values[0] if len(multi[multi['Model'] == m]) > 0 else 0 
                      for m in MODEL_TYPES]
            
            ax.bar(x - w/2, s_vals, w, label='Same-Stock', color='#0072B2', alpha=0.8)
            ax.bar(x + w/2, m_vals, w, label='Multi-Stock', color='#D55E00', alpha=0.8)
            
            ax.set_title(f'{target}', fontsize=14)
            ax.set_xticks(x)
            ax.set_xticklabels(MODEL_TYPES, fontsize=10)
            ax.set_ylabel('RMSE', fontsize=12)
            ax.legend(fontsize=9)
        
        fig.tight_layout()
        save_fig(fig, 'figures/summary/same_vs_multi_stock_rmse.png')
        print("Same vs Multi-stock comparison saved!")


  Multi-Stock Training — Results
  Train_Stocks Target_Stock  Model        MSE     RMSE      MAE  MAPE (%)       R2  Training_Time_s  Epochs_Run
BBCA+ASII+UNVR         TLKM BiLSTM  4214.8083  64.9216  48.7329    1.5994 0.975328            573.9         100
BBCA+ASII+UNVR         TLKM  BiGRU  3785.9062  61.5297  46.0793    1.5081 0.977839            527.9         100
BBCA+ASII+UNVR         TLKM   LSTM  6048.8863  77.7746  62.6780    2.0066 0.964593            324.1         100
BBCA+ASII+UNVR         TLKM    GRU 10774.4113 103.7999  90.8518    2.8870 0.936931            306.1         100
TLKM+ASII+UNVR         BBCA BiLSTM 91950.6058 303.2336 250.8468    2.9069 0.914927            674.7         100
TLKM+ASII+UNVR         BBCA  BiGRU 92272.2478 303.7635 275.7658    3.3198 0.914629            745.2         100
TLKM+ASII+UNVR         BBCA   LSTM 16873.2983 129.8973  96.6097    1.1622 0.984389            409.2         100
TLKM+ASII+UNVR         BBCA    GRU 14226.4884 119.2748  87.4111    1.06

## 7. Comprehensive Results Table

In [9]:
# ============================================================
# MASTER RESULTS TABLE
# ============================================================
print("\n" + "=" * 80)
print("  COMPREHENSIVE RESULTS SUMMARY")
print("=" * 80)

for exp_name, df in sorted(results.items()):
    print(f"\n{'─'*60}")
    print(f"  {exp_name}")
    print(f"{'─'*60}")
    
    # Best model
    if 'RMSE' in df.columns:
        valid = df.dropna(subset=['RMSE'])
        if not valid.empty:
            best_idx = valid['RMSE'].idxmin()
            best = valid.loc[best_idx]
            print(f"  Best by RMSE: {best['Model']} (RMSE={best['RMSE']:.4f})")
            
            worst_idx = valid['RMSE'].idxmax()
            worst = valid.loc[worst_idx]
            print(f"  Worst by RMSE: {worst['Model']} (RMSE={worst['RMSE']:.4f})")
    
    # Average per model
    if 'Model' in df.columns:
        avg = df.groupby('Model')[['RMSE', 'MAE', 'MAPE (%)', 'R2']].mean()
        print(f"\n  Average metrics per model:")
        print(avg.round(4).to_string())



  COMPREHENSIVE RESULTS SUMMARY

────────────────────────────────────────────────────────────
  Exp1_70_30
────────────────────────────────────────────────────────────
  Best by RMSE: BiGRU (RMSE=75.1952)
  Worst by RMSE: BiGRU (RMSE=389.2240)

  Average metrics per model:
            RMSE       MAE  MAPE (%)      R2
Model                                       
BiGRU   175.3997  140.5100    2.8117  0.9713
BiLSTM  168.1788  131.3985    2.9234  0.9665
GRU     144.2118  114.4482    2.3824  0.9788
LSTM    146.6866  113.1102    2.4700  0.9761

────────────────────────────────────────────────────────────
  Exp1_80_20
────────────────────────────────────────────────────────────
  Best by RMSE: BiGRU (RMSE=76.0569)
  Worst by RMSE: GRU (RMSE=276.2793)

  Average metrics per model:
            RMSE       MAE  MAPE (%)      R2
Model                                       
BiGRU   126.1476   98.4187    2.3301  0.9708
BiLSTM  147.5304  118.5387    2.8284  0.9613
GRU     135.7246  106.7105    2.184

## 8. Overall Winner Analysis

In [10]:
# ============================================================
# OVERALL WINNER ANALYSIS
# ============================================================
print("\n" + "=" * 80)
print("  OVERALL MODEL RANKING")
print("=" * 80)

# Collect average RMSE and R² across all experiments
model_scores = {m: {'rmse_sum': 0, 'r2_sum': 0, 'count': 0} for m in MODEL_TYPES}

for exp_name, df in results.items():
    if 'Model' not in df.columns:
        continue
    for m in MODEL_TYPES:
        model_data = df[df['Model'] == m].dropna(subset=['RMSE', 'R2'])
        if not model_data.empty:
            model_scores[m]['rmse_sum'] += model_data['RMSE'].mean()
            model_scores[m]['r2_sum'] += model_data['R2'].mean()
            model_scores[m]['count'] += 1

# Compute overall averages
overall = []
for m in MODEL_TYPES:
    if model_scores[m]['count'] > 0:
        overall.append({
            'Model': m,
            'Avg_RMSE': model_scores[m]['rmse_sum'] / model_scores[m]['count'],
            'Avg_R2': model_scores[m]['r2_sum'] / model_scores[m]['count'],
            'N_Experiments': model_scores[m]['count'],
        })

overall_df = pd.DataFrame(overall)
overall_df = overall_df.sort_values('Avg_RMSE')
print(overall_df.to_string(index=False))

# Final ranking plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Overall Model Ranking Across All Experiments',
             fontsize=16, fontweight='bold')

# By RMSE (lower is better)
ax = axes[0]
sorted_df = overall_df.sort_values('Avg_RMSE')
bars = ax.barh(sorted_df['Model'], sorted_df['Avg_RMSE'],
               color=[MODEL_COLORS[m] for m in sorted_df['Model']])
ax.set_xlabel('Average RMSE (lower is better)', fontsize=13)
ax.set_title('Ranked by RMSE', fontsize=14)
for bar, val in zip(bars, sorted_df['Avg_RMSE']):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
            f' {val:.2f}', va='center', fontsize=11)

# By R² (higher is better)
ax = axes[1]
sorted_df = overall_df.sort_values('Avg_R2', ascending=True)
bars = ax.barh(sorted_df['Model'], sorted_df['Avg_R2'],
               color=[MODEL_COLORS[m] for m in sorted_df['Model']])
ax.set_xlabel('Average R² (higher is better)', fontsize=13)
ax.set_title('Ranked by R²', fontsize=14)
for bar, val in zip(bars, sorted_df['Avg_R2']):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
            f' {val:.4f}', va='center', fontsize=11)

fig.tight_layout()
save_fig(fig, 'figures/summary/overall_model_ranking.png')

print("\n\nOverall ranking plot saved!")
print("\n" + "=" * 40)
print("  WINNER: " + overall_df.iloc[0]['Model'])
print("=" * 40)



  OVERALL MODEL RANKING
 Model   Avg_RMSE   Avg_R2  N_Experiments
   GRU 133.248004 0.971918              8
  LSTM 215.541603 0.884797              8
 BiGRU 243.638666 0.832232              8
BiLSTM 327.394476 0.586948              8
  Figure saved: figures/summary/overall_model_ranking.png


Overall ranking plot saved!

  WINNER: GRU


## 9. Conclusion

All results and figures have been saved:

**Results CSVs:** `results/` directory  
**Figures:** `figures/summary/` directory (600 DPI, PNG format)

**Key Figure Files:**
- `exp1_rmse_80vs70.png` — 80/20 vs 70/30 RMSE comparison
- `exp1_r2_80vs70.png` — 80/20 vs 70/30 R² comparison
- `bilstm_vs_bigru_same_stock.png` — Main model comparison
- `bidirectional_vs_unidirectional.png` — Bi- vs Uni-directional
- `same_vs_cross_stock_rmse.png` — Generalization analysis
- `same_vs_multi_stock_rmse.png` — Multi-stock training effect
- `overall_model_ranking.png` — Final ranking

These figures are formatted for IEEE publication at 600 DPI.
